# ==============================================================================
# UNIVERSIDAD DE LA SABANA - EICEA
# ASIGNATURA: Programación y Decisiones
# TEMA: Bases de Datos Relacionales (SQL con SQLite)
# ACTIVIDAD: Solución Ideal Tarea 1 Clase 7 - CRUD en SQLite
# ==============================================================================

# INICIAR BASES DE DATOS

In [21]:
import sqlite3

# Definimos el nombre de la base de datos como una constante global
DB_NAME = "tarea_1_cafeteria_sabana_crud.db"

def inicializar_base_datos():
    """
    Crea las tablas necesarias para el sistema de la Cafetería U. Sabana.
    Se utiliza 'IF NOT EXISTS' para no sobrescribir datos si la tabla ya existe.
    """
    # El uso de 'with' abre y cierra automáticamente la conexión, evitando que la BD se bloquee.
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        
        # Activamos el soporte para Llaves Foráneas (Obligatorio en SQLite)
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        # 1. Tabla Cliente
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS Cliente (
                id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                email TEXT UNIQUE,
                telefono TEXT,
                tipo_cliente TEXT
            )
        ''')
        
        # 2. Tabla Proveedor
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS Proveedor (
                id_proveedor INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                empresa TEXT,
                direccion TEXT,
                telefono TEXT,
                email TEXT
            )
        ''')
        
        # 3. Tabla Producto (Módulo base visto en clase)
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS Producto (
                id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                precio REAL NOT NULL,
                stock INTEGER NOT NULL
            )
        ''')
        
        # 4. Tabla Ventas / CarritoDeCompras (Con Llaves Foráneas)
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS Ventas (
                id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
                id_cliente INTEGER,
                id_producto INTEGER,
                cantidad INTEGER NOT NULL,
                total REAL NOT NULL,
                fecha TEXT DEFAULT CURRENT_TIMESTAMP,
                FOREIGN KEY (id_cliente) REFERENCES Cliente(id_cliente),
                FOREIGN KEY (id_producto) REFERENCES Producto(id_producto)
            )
        ''')
        
        conn.commit()

In [22]:
inicializar_base_datos()

print("✅ Base de datos y tablas inicializadas correctamente.")

print("""
💡 TIP DE AUTOGESTIÓN:
Recuerda presionar "Shift + Control + P" para ir a la barra de búsqueda de comandos y ejecutar ">SQL: Open Database" para abrir la base de datos y verificar la estructura de las tablas.
""")

✅ Base de datos y tablas inicializadas correctamente.

💡 TIP DE AUTOGESTIÓN:
Recuerda presionar "Shift + Control + P" para ir a la barra de búsqueda de comandos y ejecutar ">SQL: Open Database" para abrir la base de datos y verificar la estructura de las tablas.



# ==========================================
# MÓDULO 1: OPERACIONES CRUD - CLIENTE
# ==========================================

In [23]:
# CREATE
def crear_cliente(nombre, email, telefono, tipo_cliente):
    try:
        with sqlite3.connect(DB_NAME) as conn:
            cursor = conn.cursor()
            cursor.execute('''
                INSERT INTO Cliente (nombre, email, telefono, tipo_cliente) 
                VALUES (?, ?, ?, ?)
            ''', (nombre, email, telefono, tipo_cliente))
            conn.commit()
            print(f"➕ Cliente '{nombre}' creado exitosamente.")
    except sqlite3.IntegrityError:
        print(f"⚠️ Error: El email '{email}' ya se encuentra registrado.")

# READ
def leer_clientes():
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Cliente")
        resultados = cursor.fetchall()
        print("\n👥 LISTA DE CLIENTES:")
        for fila in resultados:
            print(f"ID: {fila[0]} | Nombre: {fila[1]} | Email: {fila[2]} | Tel: {fila[3]} | Tipo: {fila[4]}")
        print("-" * 50)

# UPDATE
def actualizar_telefono_cliente(id_cliente, nuevo_telefono):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            UPDATE Cliente 
            SET telefono = ? 
            WHERE id_cliente = ?
        ''', (nuevo_telefono, id_cliente))
        conn.commit()
        print(f"🔄 Teléfono actualizado para el Cliente ID {id_cliente}.")

# DELETE
def eliminar_cliente(id_cliente):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Cliente WHERE id_cliente = ?", (id_cliente,))
        conn.commit()
        print(f"❌ Cliente ID {id_cliente} eliminado del sistema.")

In [24]:
print("--- PRUEBAS MÓDULO CLIENTE ---")
crear_cliente("Diego Zuluaga", "diego@unisabana.edu.co", "3001112233", "Profesor")
crear_cliente("Ana Gómez", "ana@unisabana.edu.co", "3109998877", "Estudiante")

leer_clientes()

actualizar_telefono_cliente(1, "3200000000")
leer_clientes()

eliminar_cliente(2)
leer_clientes()

--- PRUEBAS MÓDULO CLIENTE ---
➕ Cliente 'Diego Zuluaga' creado exitosamente.
⚠️ Error: El email 'ana@unisabana.edu.co' ya se encuentra registrado.

👥 LISTA DE CLIENTES:
ID: 1 | Nombre: Estudiante Prueba | Email: prueba@correo.com | Tel: 3200000000 | Tipo: Estudiante
ID: 3 | Nombre: Ana Gómez | Email: ana@unisabana.edu.co | Tel: 3109998877 | Tipo: Estudiante
ID: 4 | Nombre: Diego Zuluaga | Email: diego@unisabana.edu.co | Tel: 3001112233 | Tipo: Profesor
--------------------------------------------------
🔄 Teléfono actualizado para el Cliente ID 1.

👥 LISTA DE CLIENTES:
ID: 1 | Nombre: Estudiante Prueba | Email: prueba@correo.com | Tel: 3200000000 | Tipo: Estudiante
ID: 3 | Nombre: Ana Gómez | Email: ana@unisabana.edu.co | Tel: 3109998877 | Tipo: Estudiante
ID: 4 | Nombre: Diego Zuluaga | Email: diego@unisabana.edu.co | Tel: 3001112233 | Tipo: Profesor
--------------------------------------------------
❌ Cliente ID 2 eliminado del sistema.

👥 LISTA DE CLIENTES:
ID: 1 | Nombre: Estudiant

# ==========================================
# MÓDULO 2: OPERACIONES CRUD - PROVEEDOR
# ==========================================

In [25]:
# CREATE
def crear_proveedor(nombre, empresa, direccion, telefono, email):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO Proveedor (nombre, empresa, direccion, telefono, email) 
            VALUES (?, ?, ?, ?, ?)
        ''', (nombre, empresa, direccion, telefono, email))
        conn.commit()
        print(f"➕ Proveedor '{empresa}' ({nombre}) creado exitosamente.")

# READ
def leer_proveedores():
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Proveedor")
        resultados = cursor.fetchall()
        print("\n🏢 LISTA DE PROVEEDORES:")
        for fila in resultados:
            print(f"ID: {fila[0]} | Empresa: {fila[2]} | Contacto: {fila[1]} | Tel: {fila[4]}")
        print("-" * 50)

# UPDATE
def actualizar_direccion_proveedor(id_proveedor, nueva_direccion):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            UPDATE Proveedor 
            SET direccion = ? 
            WHERE id_proveedor = ?
        ''', (nueva_direccion, id_proveedor))
        conn.commit()
        print(f"🔄 Dirección actualizada para el Proveedor ID {id_proveedor}.")

# DELETE
def eliminar_proveedor(id_proveedor):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Proveedor WHERE id_proveedor = ?", (id_proveedor,))
        conn.commit()
        print(f"❌ Proveedor ID {id_proveedor} eliminado del sistema.")

In [26]:
print("--- PRUEBAS MÓDULO PROVEEDOR ---")

crear_proveedor("Carlos Ruiz", "Distribuidora Sabana", "Calle 123", "3005554433", "contacto@dsabana.com")
crear_proveedor("María López", "Insumos Panaderos", "Carrera 45", "3112223344", "ventas@ipanaderos.com")

leer_proveedores()

actualizar_direccion_proveedor(1, "Avenida Central #45-67")
leer_proveedores()

eliminar_proveedor(2)
leer_proveedores()

--- PRUEBAS MÓDULO PROVEEDOR ---
➕ Proveedor 'Distribuidora Sabana' (Carlos Ruiz) creado exitosamente.
➕ Proveedor 'Insumos Panaderos' (María López) creado exitosamente.

🏢 LISTA DE PROVEEDORES:
ID: 1 | Empresa: Distribuidora Sabana | Contacto: Carlos Ruiz | Tel: 3005554433
ID: 3 | Empresa: Distribuidora Sabana | Contacto: Carlos Ruiz | Tel: 3005554433
ID: 4 | Empresa: Insumos Panaderos | Contacto: María López | Tel: 3112223344
--------------------------------------------------
🔄 Dirección actualizada para el Proveedor ID 1.

🏢 LISTA DE PROVEEDORES:
ID: 1 | Empresa: Distribuidora Sabana | Contacto: Carlos Ruiz | Tel: 3005554433
ID: 3 | Empresa: Distribuidora Sabana | Contacto: Carlos Ruiz | Tel: 3005554433
ID: 4 | Empresa: Insumos Panaderos | Contacto: María López | Tel: 3112223344
--------------------------------------------------
❌ Proveedor ID 2 eliminado del sistema.

🏢 LISTA DE PROVEEDORES:
ID: 1 | Empresa: Distribuidora Sabana | Contacto: Carlos Ruiz | Tel: 3005554433
ID: 3 | Emp

# ==========================================
# MÓDULO 3: OPERACIONES CRUD - PRODUCTO
# ==========================================

In [27]:
# CREATE
def crear_producto(nombre, precio, stock):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO Producto (nombre, precio, stock) 
            VALUES (?, ?, ?)
        ''', (nombre, precio, stock))
        conn.commit()
        print(f"➕ Producto '{nombre}' creado exitosamente.")

# READ
def leer_productos():
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Producto")
        resultados = cursor.fetchall()
        print("\n📦 LISTA DE PRODUCTOS:")
        for fila in resultados:
            print(f"ID: {fila[0]} | Nombre: {fila[1]} | Precio: ${fila[2]:,.2f} | Stock: {fila[3]}")
        print("-" * 50)

# UPDATE
def actualizar_precio_producto(id_producto, nuevo_precio):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            UPDATE Producto 
            SET precio = ? 
            WHERE id_producto = ?
        ''', (nuevo_precio, id_producto))
        conn.commit()
        print(f"🔄 Precio actualizado para el Producto ID {id_producto}.")

# DELETE
def eliminar_producto(id_producto):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Producto WHERE id_producto = ?", (id_producto,))
        conn.commit()
        print(f"❌ Producto ID {id_producto} eliminado del sistema.")

In [28]:
print("--- PRUEBAS MÓDULO PRODUCTO ---")

crear_producto("Café Tostao", 5000.0, 50)
crear_producto("Chocolatina Jet", 1200.0, 100)

leer_productos()

actualizar_precio_producto(1, 5500.0)
leer_productos()

eliminar_producto(2)
leer_productos()

--- PRUEBAS MÓDULO PRODUCTO ---
➕ Producto 'Café Tostao' creado exitosamente.
➕ Producto 'Chocolatina Jet' creado exitosamente.

📦 LISTA DE PRODUCTOS:
ID: 1 | Nombre: Empanada | Precio: $5,500.00 | Stock: 20
ID: 3 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
ID: 4 | Nombre: Café Tostao | Precio: $5,000.00 | Stock: 50
ID: 5 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
--------------------------------------------------
🔄 Precio actualizado para el Producto ID 1.

📦 LISTA DE PRODUCTOS:
ID: 1 | Nombre: Empanada | Precio: $5,500.00 | Stock: 20
ID: 3 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
ID: 4 | Nombre: Café Tostao | Precio: $5,000.00 | Stock: 50
ID: 5 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
--------------------------------------------------
❌ Producto ID 2 eliminado del sistema.

📦 LISTA DE PRODUCTOS:
ID: 1 | Nombre: Empanada | Precio: $5,500.00 | Stock: 20
ID: 3 | Nombre: Chocolatina Jet | Precio: $1,200.00 | Stock: 100
ID

# ==========================================
# PREPARACIÓN PARA MÓDULO VENTAS (LIMPIEZA DE BD)
# ==========================================

In [29]:
def limpiar_base_datos():
    """
    Elimina todos los datos y reinicia los contadores de AUTOINCREMENT.
    Esto soluciona el problema de los IDs que cambian en cada ejecución de prueba,
    permitiendo que los UPDATE y DELETE por ID funcionen de manera predecible.
    """
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        tablas =['Ventas', 'Producto', 'Proveedor', 'Cliente']
        
        for tabla in tablas:
            # Borramos los registros
            cursor.execute(f"DELETE FROM {tabla}")
            # Reiniciamos el contador de AUTOINCREMENT en la tabla interna de SQLite
            cursor.execute("DELETE FROM sqlite_sequence WHERE name=?", (tabla,))
        
        conn.commit()
    print("🧹 Base de datos limpiada y contadores de ID reiniciados a 1.")

def setup_datos_iniciales():
    """
    Inserta datos base para que la tabla Ventas tenga Llaves Foráneas válidas.
    """
    crear_cliente("Estudiante Prueba", "prueba@correo.com", "0000", "Estudiante")
    crear_producto("Empanada", 3000.0, 20)
    print("⚙️ Datos iniciales de Cliente (ID 1) y Producto (ID 1) configurados.")


# ==========================================
# MÓDULO 4: OPERACIONES CRUD - VENTAS (CARRITO)
# ==========================================

In [30]:
# CREATE
def crear_venta(id_cliente, id_producto, cantidad, total):
    
    # Se usa with para asegurar que la conexión se cierre correctamente, evitando bloqueos en la base de datos.
    
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;") # Activamos FK para esta transacción
        try:
            cursor.execute('''
                INSERT INTO Ventas (id_cliente, id_producto, cantidad, total) 
                VALUES (?, ?, ?, ?)
            ''', (id_cliente, id_producto, cantidad, total))
            conn.commit()
            print(f"➕ Venta registrada exitosamente. Total: ${total:,.2f}")
        except sqlite3.IntegrityError:
            print("⚠️ Error: El id_cliente o id_producto no existe en la base de datos.")

# READ
def leer_ventas():
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM Ventas")
        resultados = cursor.fetchall()
        print("\n🧾 REGISTRO DE VENTAS:")
        for fila in resultados:
            print(f"ID Venta: {fila[0]} | ID Cliente: {fila[1]} | ID Prod: {fila[2]} | Cant: {fila[3]} | Total: ${fila[4]:,.2f} | Fecha: {fila[5]}")
        print("-" * 50)

# UPDATE
def actualizar_cantidad_venta(id_venta, nueva_cantidad, nuevo_total):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            UPDATE Ventas 
            SET cantidad = ?, total = ? 
            WHERE id_venta = ?
        ''', (nueva_cantidad, nuevo_total, id_venta))
        conn.commit()
        print(f"🔄 Venta ID {id_venta} actualizada. Nueva cantidad: {nueva_cantidad}, Nuevo Total: ${nuevo_total:,.2f}")

# DELETE
def eliminar_venta(id_venta):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("DELETE FROM Ventas WHERE id_venta = ?", (id_venta,))
        conn.commit()
        print(f"❌ Venta ID {id_venta} eliminada del sistema.")


In [33]:
print("--- PRUEBAS MÓDULO VENTAS ---")

# 1. Limpiamos y preparamos el entorno para que los IDs sean exactamente 1
limpiar_base_datos()
setup_datos_iniciales()

# 2. Ejecutamos las pruebas CRUD
print("\n[Ejecutando CRUD Ventas]")
# Insertamos una venta válida (Cliente 1 compra 2 Empanadas a 3000 c/u)
crear_venta(id_cliente=1, id_producto=1, cantidad=2, total=6000.0)

# Intentamos insertar una venta con un Cliente que no existe (Prueba de Integridad FK)
crear_venta(id_cliente=99, id_producto=1, cantidad=1, total=3000.0)

leer_ventas()

# Actualizamos la venta (Ahora compró 3 empanadas)
actualizar_cantidad_venta(id_venta=1, nueva_cantidad=3, nuevo_total=9000.0)
leer_ventas()

# Eliminamos la venta
eliminar_venta(id_venta=1)
leer_ventas()

## Insertamos nuevamente para verificar que el ID se reinició a 1

# Reiniciamos el contador de AUTOINCREMENT para la tabla Ventas
with sqlite3.connect(DB_NAME) as conn:
    cursor = conn.cursor()
    cursor.execute("DELETE FROM sqlite_sequence WHERE name='Ventas'")
    conn.commit()
    print("🔄 Contador de ID para Ventas reiniciado a 1.")

crear_venta(id_cliente=1, id_producto=1, cantidad=1, total=3000.0)
leer_ventas()

--- PRUEBAS MÓDULO VENTAS ---
🧹 Base de datos limpiada y contadores de ID reiniciados a 1.
➕ Cliente 'Estudiante Prueba' creado exitosamente.
➕ Producto 'Empanada' creado exitosamente.
⚙️ Datos iniciales de Cliente (ID 1) y Producto (ID 1) configurados.

[Ejecutando CRUD Ventas]
➕ Venta registrada exitosamente. Total: $6,000.00
⚠️ Error: El id_cliente o id_producto no existe en la base de datos.

🧾 REGISTRO DE VENTAS:
ID Venta: 1 | ID Cliente: 1 | ID Prod: 1 | Cant: 2 | Total: $6,000.00 | Fecha: 2026-04-08 21:31:08
--------------------------------------------------
🔄 Venta ID 1 actualizada. Nueva cantidad: 3, Nuevo Total: $9,000.00

🧾 REGISTRO DE VENTAS:
ID Venta: 1 | ID Cliente: 1 | ID Prod: 1 | Cant: 3 | Total: $9,000.00 | Fecha: 2026-04-08 21:31:08
--------------------------------------------------
❌ Venta ID 1 eliminada del sistema.

🧾 REGISTRO DE VENTAS:
--------------------------------------------------
🔄 Contador de ID para Ventas reiniciado a 1.
➕ Venta registrada exitosamente.